# Lab report

This notebook builds outputs/lab_report.pdf from the charts saved by multivariate_analysis.ipynb. Run the analysis notebook first so the figure files exist.


In [1]:
# Build the PDF from the charts the analysis notebook already saved.
from pathlib import Path

from fpdf import FPDF

ROOT = Path.cwd()
FIG_DIR = ROOT / "outputs" / "figures"

chi_square = 1352.0
bartlett_p = 9.840e-261
kmo_overall = 0.782

TAKEAWAYS = {
    "scatter_matrix": {
        "takeaway": "Electricity access rises with income (correlation 0.74), while renewable energy is more scattered and does not follow income as tightly.",
        "decision": "The matrix shows only four indicators, in the lower triangle, so a general reader can see the pairs without a crowded 9-by-9 grid.",
    },
    "correlation_heatmap": {
        "takeaway": "Electricity access and income per person move together (correlation 0.74), and both move opposite to renewable energy share.",
        "decision": "A red-blue scale centered at zero makes positive and negative relationships visible without reading every number.",
    },
    "pca_scree": {
        "takeaway": "The first 3 components keep at least 80% of the variation. The first two alone keep 69.7%.",
        "decision": "An 80% line on the scree plot makes the cutoff visible instead of relying on a table of eigenvalues alone.",
    },
    "pca_biplot": {
        "takeaway": "Component 1 contrasts Energy use per person (log) with Renewable energy. Together the first two components keep 69.7% of the variation.",
        "decision": "Arrows are drawn on the same axes as the countries, and each arrow is labeled with a plain-language indicator name.",
    },
    "factor_loadings": {
        "takeaway": "Bartlett's test is significant (p = 9.84e-261) and KMO is 0.78, so 3 hidden factors are a reasonable summary. Factor 1 is dominated by Energy use per person (log).",
        "decision": "Varimax rotation and Kaiser's eigenvalue rule were used so each indicator mostly belongs to one readable factor.",
    },
    "lda_discriminant": {
        "takeaway": "Energy indicators alone place 91% of countries on the correct side of the median-income split. CO2 per person (log) carries the most weight.",
        "decision": "The groups are split at $6,095 per person, and income is left out of the predictors so the model cannot simply rediscover the label.",
    },
    "parallel_coordinates": {
        "takeaway": "Higher-income countries sit above average on electricity access, clean cooking fuels, and energy use, while lower-income countries sit higher on renewable energy.",
        "decision": "Each indicator was converted to a common scale before plotting, so a percentage is not visually overpowered by a larger raw number.",
    },
    "trellis_decade": {
        "takeaway": "From the 2000s to the 2010s, more country-years sit at high electricity access, while renewable energy stays spread out even among well-connected countries.",
        "decision": "Splitting the panel by decade keeps the time pattern visible without putting twenty years on one crowded scatter plot.",
    },
    "income_radar_glyph": {
        "takeaway": "The higher-income icon stretches toward electricity access, clean cooking fuels, and energy use, while the lower-income icon stretches toward renewable energy.",
        "decision": "Medians were rescaled spoke by spoke so the star chart compares profile shape instead of mixing percentages with log values.",
    },
}

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=14)
pdf.add_page()


def write_line(text: str, size: int = 11, bold: bool = False) -> None:
    pdf.set_font("Helvetica", "B" if bold else "", size)
    pdf.set_x(pdf.l_margin)
    pdf.multi_cell(pdf.epw, 6, text, new_x="LMARGIN", new_y="NEXT")


write_line("Multivariate Energy Analysis Lab", size=16, bold=True)
write_line(
    "Country-year energy indicators, 2000-2019. Factor analysis, PCA, and discriminant analysis use the latest year for each country."
)
pdf.ln(2)
write_line("Questions", size=12, bold=True)
write_line("1. Can a few hidden factors summarize energy access, renewables, intensity, emissions, and income?")
write_line("2. How many combined scores keep most of the variation in these indicators?")
write_line("3. Which energy indicators, excluding income itself, best separate higher- and lower-income countries?")
pdf.ln(2)
write_line("Preliminary tests", size=12, bold=True)
write_line(
    f"Bartlett's test chi-square = {chi_square:,.1f}, p-value = {bartlett_p:.3e}. "
    f"A p-value below 0.05 means the indicators are correlated enough for factor analysis. "
    f"KMO = {kmo_overall:.3f}. A KMO above 0.60 supports a factor model."
)

# One page per chart, with the takeaway and the choice we made.
sections = [
    ("How key energy indicators relate to each other", "scatter_matrix"),
    ("Which energy and income indicators move together", "correlation_heatmap"),
    ("How many summary scores are worth keeping", "pca_scree"),
    ("Country scores and the indicators that pull them", "pca_biplot"),
    ("Which indicators share the same hidden factor", "factor_loadings"),
    ("How well energy indicators separate the two income groups", "lda_discriminant"),
    ("Country profiles across energy indicators", "parallel_coordinates"),
    ("Electricity access and renewable energy, by decade", "trellis_decade"),
    ("Typical energy profile of each income group", "income_radar_glyph"),
]
for title, key in sections:
    pdf.add_page()
    write_line(title, size=12, bold=True)
    pdf.ln(1)
    write_line("Takeaway: " + TAKEAWAYS[key]["takeaway"])
    pdf.ln(1)
    write_line("Design decision: " + TAKEAWAYS[key]["decision"])
    pdf.ln(3)
    pdf.image(str(FIG_DIR / f"{key}.png"), w=pdf.epw)

report_path = ROOT / "outputs" / "lab_report.pdf"
pdf.output(str(report_path))
print(f"Saved {report_path}")


Saved C:\Users\Alden Olmedo\Documents\VSCode\multivariate-data-analysis\outputs\lab_report.pdf
